# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
import joblib
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from itertools import product
from tqdm import tqdm

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [43]:
df = pd.concat([pd.read_csv('../data/day-of-week-not-scaled.csv'), pd.read_csv('../data/dayofweek.csv')['dayofweek']], axis=1)
X = df.drop(columns='dayofweek')
y = df['dayofweek']

In [44]:
df.columns

Index(['numTrials', 'hour', 'uid_user_0', 'uid_user_1', 'uid_user_10',
       'uid_user_11', 'uid_user_12', 'uid_user_13', 'uid_user_14',
       'uid_user_15', 'uid_user_16', 'uid_user_17', 'uid_user_18',
       'uid_user_19', 'uid_user_2', 'uid_user_20', 'uid_user_21',
       'uid_user_22', 'uid_user_23', 'uid_user_24', 'uid_user_25',
       'uid_user_26', 'uid_user_27', 'uid_user_28', 'uid_user_29',
       'uid_user_3', 'uid_user_30', 'uid_user_31', 'uid_user_4', 'uid_user_6',
       'uid_user_7', 'uid_user_8', 'labname_code_rvw', 'labname_lab02',
       'labname_lab03', 'labname_lab03s', 'labname_lab05s', 'labname_laba04',
       'labname_laba04s', 'labname_laba05', 'labname_laba06',
       'labname_laba06s', 'labname_project1', 'dayofweek'],
      dtype='object')

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
svc = SVC(random_state=21, probability=True)

param_grid = {
    'kernel' : ['linear', 'rbf', 'sigmoid'],
    'C' : [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma' : ['scale', 'auto'],
    'class_weight':['balanced', 'None']
}

grid_search = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1             
)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print(f"Лучшая Accuracy на кросс-валидации: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_ 
test_accuracy = best_model.score(X_test, y_test)
print(f"Accuracy лучшей модели на тесте: {test_accuracy:.4f}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    4.0s
[Parallel(n_jobs=-1)]: Done 231 tasks      | elapsed:   31.5s
[Parallel(n_jobs=-1)]: Done 360 out of 360 | elapsed:  2.5min finished


Лучшие параметры: {'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf'}
Лучшая Accuracy на кросс-валидации: 0.8635
Accuracy лучшей модели на тесте: 0.8876


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values(by='rank_test_score', ascending=True)

cols_to_show = [
    'rank_test_score', 
    'mean_test_score', 
    'std_test_score', 
    'param_kernel', 
    'param_C', 
    'param_gamma', 
    'param_class_weight',
    'mean_fit_time'
]

results_df[cols_to_show].head(10)

,rank_test_score,mean_test_score,std_test_score,param_kernel,param_C,param_gamma,param_class_weight,mean_fit_time
64,1,0.863500,0.010870,rbf,10,auto,balanced,0.640861
52,2,0.808608,0.021007,rbf,5,auto,balanced,0.558701
60,3,0.721052,0.034438,linear,10,scale,balanced,52.356289
63,3,0.721052,0.034438,linear,10,auto,balanced,36.823824
48,5,0.706234,0.031619,linear,5,scale,balanced,33.565330
51,5,0.706234,0.031619,linear,5,auto,balanced,32.172358
36,7,0.665419,0.016680,linear,1.5,scale,balanced,15.446006
39,7,0.665419,0.016680,linear,1.5,auto,balanced,14.978552
27,9,0.638720,0.021487,linear,1,auto,balanced,11.878927
24,9,0.638720,0.021487,linear,1,scale,balanced,11.699460


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
dt_model = DecisionTreeClassifier(random_state=21)

param_grid = {
    'max_depth' : range(1, 50),
    'class_weight':['balanced', 'None'],
    'criterion':['entropy', 'gini']
}
grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1             
)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print(f"Лучшая Accuracy на кросс-валидации: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Accuracy лучшей модели на тесте: {test_accuracy:.4f}")

Fitting 5 folds for each of 196 candidates, totalling 980 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done 725 tasks      | elapsed:    3.0s


Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}
Лучшая Accuracy на кросс-валидации: 0.8739
Accuracy лучшей модели на тесте: 0.8846


[Parallel(n_jobs=-1)]: Done 980 out of 980 | elapsed:    3.3s finished


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values(by='rank_test_score', ascending=True)

cols_to_show = [
    'rank_test_score', 
    'mean_test_score', 
    'std_test_score', 
    'param_max_depth', 
    'param_criterion', 
    'param_class_weight',
    'mean_fit_time'
]

results_df[cols_to_show].head(10)

,rank_test_score,mean_test_score,std_test_score,param_max_depth,param_criterion,param_class_weight,mean_fit_time
69,1,0.873865,0.025066,21,gini,balanced,0.006558
73,2,0.873854,0.025018,25,gini,balanced,0.006157
70,3,0.872378,0.025263,22,gini,balanced,0.007278
97,4,0.872372,0.025179,49,gini,balanced,0.006198
71,4,0.872372,0.025179,23,gini,balanced,0.006197
75,4,0.872372,0.025179,27,gini,balanced,0.006752
76,4,0.872372,0.025179,28,gini,balanced,0.013014
77,4,0.872372,0.025179,29,gini,balanced,0.006797
78,4,0.872372,0.025179,30,gini,balanced,0.007201
79,4,0.872372,0.025179,31,gini,balanced,0.005997


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
rf_model = RandomForestClassifier(random_state=21)

grid_param = {
    'n_estimators' : [5, 10, 50, 100],
    'max_depth' : range(1, 50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini']
}

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=grid_param,
    scoring='accuracy',   
    n_jobs=-1,            
    verbose=1             
)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print(f"Лучшая Accuracy на кросс-валидации: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Accuracy лучшей модели на тесте: {test_accuracy:.4f}")

Fitting 5 folds for each of 784 candidates, totalling 3920 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  31 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 328 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done 828 tasks      | elapsed:   20.3s
[Parallel(n_jobs=-1)]: Done 1528 tasks      | elapsed:   35.1s
[Parallel(n_jobs=-1)]: Done 2428 tasks      | elapsed:   54.3s
[Parallel(n_jobs=-1)]: Done 3528 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 3920 out of 3920 | elapsed:  1.4min finished


Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}
Лучшая Accuracy на кросс-валидации: 0.9043
Accuracy лучшей модели на тесте: 0.9260


In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values(by='rank_test_score', ascending=True)

cols_to_show = [
    'rank_test_score', 
    'mean_test_score', 
    'std_test_score', 
    'param_max_depth', 
    'param_n_estimators',
    'param_criterion', 
    'param_class_weight',
    'mean_fit_time'
]

results_df[cols_to_show].head(10)

,rank_test_score,mean_test_score,std_test_score,param_max_depth,param_n_estimators,param_criterion,param_class_weight,mean_fit_time
95,1,0.904293,0.012361,24,100,entropy,balanced,0.651401
115,2,0.904290,0.012156,29,100,entropy,balanced,0.722201
698,2,0.904290,0.010961,28,50,gini,None,0.237894
314,4,0.903549,0.012056,30,50,gini,balanced,0.310532
711,5,0.903547,0.014380,31,100,gini,None,0.538653
99,6,0.902809,0.013639,25,100,entropy,balanced,0.582576
326,7,0.902809,0.013628,33,50,gini,balanced,0.301833
767,8,0.902806,0.010460,45,100,gini,None,0.498850
779,8,0.902806,0.010460,48,100,gini,None,0.507007
775,8,0.902806,0.010460,47,100,gini,None,0.595890


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
grid_param = {
    'n_estimators' : [5, 10, 50, 100],
    'max_depth' : range(1, 50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini']
}

keys = grid_param.keys()
all_combinations = [dict(zip(keys, v)) for v in product(*grid_param.values())]

results = []

for params in tqdm(all_combinations, desc="Grid Search Progress"):
    model = RandomForestClassifier(random_state=21, **params, n_jobs=-1)

    cvs = cross_val_score(estimator=model, X=X_train, y=y_train, cv=5, scoring='accuracy', n_jobs=-1)

    res = params.copy()
    res['mean_score'] = cvs.mean()
    res['std_score'] = cvs.std()
    
    results.append(res)

results_df = pd.DataFrame(results)
results_df['rank'] = results_df['mean_score'].rank(ascending=False, method='min').astype(int)
results_df = results_df.sort_values(by='rank')

print("Топ-10 лучших комбинаций параметров:")
display(results_df.head(10))

Grid Search Progress: 100%|██████████| 784/784 [05:03<00:00,  2.59it/s]

Топ-10 лучших комбинаций параметров:


,n_estimators,max_depth,class_weight,criterion,mean_score,std_score,rank
680,100,24,balanced,entropy,0.904293,0.012361,1
700,100,29,balanced,entropy,0.904290,0.012156,2
503,50,28,None,gini,0.904290,0.010961,2
509,50,30,balanced,gini,0.903549,0.012056,4
711,100,31,None,gini,0.903547,0.014380,5
684,100,25,balanced,entropy,0.902809,0.013639,6
521,50,33,balanced,gini,0.902809,0.013628,7
783,100,49,None,gini,0.902806,0.010460,8
755,100,42,None,gini,0.902806,0.010460,8
751,100,41,None,gini,0.902806,0.010460,8


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
best_model = RandomForestClassifier(n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy')
best_model.fit(X_train, y_train)
accuracy_score(y_test, best_model.predict(X_test))

0.9319526627218935